# SFT（Supervised Fine-Tuning，有监督微调）

## BERT 意图识别：从原始数据到模型输入的 5 大工序

---

### 1. 原材料（原始数据 - Raw Data）
在 `train.txt` 数据集中，每一行都是一个典型的“题目+答案”组合。
* **示例行**：`我想听周杰伦的歌    3`
* **文本 (Content)**：`我想听周杰伦的歌` —— 用户输入的原始语句。
* **分隔符**：中间是一个不可见的 **Tab 键 (`\t`)**。
* **标签 (Label)**：`3` —— 标准答案（例如：3 代表“音乐”意图）。

`self.class_list = [x.strip() for x in open(f'./data/{dataset}/class.txt', encoding='utf-8').readlines()]`
```python
class_list = []
with open('./data/intent/class.txt', encoding='utf-8') as f:
    for line in f.readlines():
        clean_line = line.strip()
        class_list.append(clean_line)
```
`12:打开空调自动模式:Open_Air_Condition_Auto_Mode`
经过 `self.class_list = [x.strip() for x in ...] `这一行代码处理后，它会发生以下变化：
- 处理前（读取时）：是一个带换行符的字符串："12:打开空调自动模式:Open_Air_Condition_Auto_Mode\n"
- .strip() 处理后：末尾看不见的 \n 被切掉了，变成干净的："12:打开空调自动模式:Open_Air_Condition_Auto_Mode"
```python
# self.class_list
[
    '0:打开路况信息:Open_Cruise_Information',
    '1:导航搜索:Go_POI',
    ...
    '12:打开空调自动模式:Open_Air_Condition_Auto_Mode',
    ...
]
```
---

### 2. 切分工序 (Tokenization)
BERT 不认识整句，需要将句子拆解为最小单元。

#### 数据预处理：Tokenizer 的切词逻辑

**1. 过程分解**：
* **输入**：`"打开空调自动模式"`
* **第一步 (Tokenize)**：拆解为 `['打', '开', '空', '调', '自', '动', '模', '式']`
* **第二步 (Add Special Tokens)**：加上 BERT 专属的班长和卫兵 `[CLS]` 和 `[SEP]`
* **第三步 (Convert to IDs)**：查字典变成数字，如 `[101, 21, 45, 12, 8, 3, 5, 9, 10, 102]`

**2. 配置体现**：
* `self.tokenizer = BertTokenizer.from_pretrained(self.bert_path)`：负责执行上述所有翻译动作。
* `self.pad_size = 32`：决定了翻译出来的数字序列最后要补多少个 `0`。

---

### 3. 数字转换 (Numericalization)
计算机只认识数字。程序通过查阅 `vocab.txt`（BERT 大字典）进行转换。这里的vocab就相当是`gpt token`模型可以听懂的最小语义单元
* **映射规则**：
    * `[CLS]` $\rightarrow$ `101`
    * `我` $\rightarrow$ `704`
    * `想` $\rightarrow$ `2395`
* **转换后序列**：`[101, 704, 2395, 110, ...]`。

---

### 4. 标准化对齐 (Padding & Masking)
为了批量处理，所有输入序列的长度必须整齐划一（如 `pad_size = 32`）。

* **Padding (补齐)**：短句子后面用 `0`（即 `[PAD]`）补齐到 32 位。
* **Mask (掩码)**：生成一个 0/1 序列。
    * **1**：代表真实文字，模型需要重点关注。
    * **0**：代表补位的废话，模型计算时自动忽略。



---

### 5. 装箱分批 (Batching)
数据量大时，通过 `DatasetIterater` 进行分块处理。
* **逻辑**：按 `batch_size`（如 128 条）将数据装入“容器”。
* **训练循环**：模型每次读取一个 Batch，对比预测结果与标签 `3` 的差距，利用优化器（如 BertAdam）微调内部参数。

## BERT 核心配置：config.json 属性全解析

### 1. 维度与容量 (Capacity)

**`"hidden_size": 312`**
* **意义**：隐藏层维度（语义空间厚度）。
* **作用**：每个字会被转换成 312 个特征值。这是 `bert_tiny` 的标准规格。

**`"vocab_size": 8021`**
* **意义**：词表总数。
* **作用**：模型“字典”里存了 8021 个字。超出范围的字会被识别为 `[UNK]`。

**`"intermediate_size": 1248`**
* **意义**：前馈网络（FFN）中间层维度。
* **作用**：模型计算时，会先从 312 维投影到 1248 维进行高维变换，再压缩回 312 维。

---

### 2. 注意力机制 (Attention)

**`"num_attention_heads": 12`**
* **意义**：多头注意力的“头”数。
* **作用**：模型同时从 12 个不同维度（如：动作、物体、语境）观察句子。

**`"attention_probs_dropout_prob": 0.1`**
* **意义**：注意力权重的随机丢弃率。
* **作用**：训练时随机“关闭” 10% 的关联，防止模型产生过拟合。

---

### 3. 结构深度 (Architecture)

**`"num_hidden_layers": 4`**
* **意义**：Transformer 层的数量（垂直深度）。
* **作用**：数据会连续经过 4 次相同的处理流程。层数越少，推理越快。

**`"max_position_embeddings": 512`**
* **意义**：最大位置编码限制。
* **作用**：定义了模型理论上能处理的最长文本极限是 512 个字符。

---

### 4. 激活与归一化 (Activation)
**`"hidden_act": "gelu"`**
* **意义**：隐藏层激活函数。
* **作用**：使用 GELU 函数实现非线性变换，是 BERT 理解复杂逻辑的关键。

---
###
**工程约束**：`bert_tiny.py` 中定义的 `hidden_size` 必须与 `config.json` 保持一致（即 312）。如果两者不匹配，在加载 `pytorch_model.bin` 权重文件时会抛出维度不符的异常。

## BERT 训练核心参数解析 (Four-Tuple)
在执行 `train_data, dev_data, test_data = build_dataset(config)` 后，每一条送入模型的数据都会被打包成以下结构：
| 参数名称 | 含义 | 物理表现 | 核心作用 |
| :--- | :--- | :--- | :--- |
| **`token_ids`** | **身份 ID 序列** | 一串数字，如 `[101, 704, 2395, ...]` | 告诉模型“这句话里有哪些字”，是语义提取的源头。 |
| **`label`** | **分类标签** | 类别索引（整数），如 `3` | 训练时的“标准答案”，用来告诉模型猜得对不对。 |
| **`seq_len`** | **实际长度** | 句子在 Padding 之前的真实字数 | 告知模型哪些是用户真话，哪些是凑数的占位符。 |
| **`mask`** | **注意力掩码** | 0/1 序列，如 `[1, 1, 1, 0, 0]` | 充当“眼罩”，强制模型忽略 `0`（Padding 位）的干扰。 |

## forward
```python
    def forward(self, x):
        context = x[0]
        mask = x[2]
        outputs = self.bert(context, attention_mask=mask, output_all_encoded_layers=False)
        last_hidden_state = outputs[0]
        cls_feature = last_hidden_state[:, 0, :]
        out = self.fc(cls_feature)
        return out
```

在切词阶段`helper.py`中，数据是这样打包的`return (x, seq_len, mask),y`,这里的输入`x`是这个三元组
### 首位的数学信息量

`cls_feature = last_hidden_state[:, 0, :]`

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

[CLS] 在这一层输出的新特征向量（我们叫它 $z_0$），是全句所有词的值向量 $v_j$，乘以对应的权重 $\alpha_{0,j}$ 后的加权求和：

$$z_0 = \sum_{j=0}^{L-1} \alpha_{0,j} v_j$$

在数学本质上$z_0$是整个 $128 \times 32 \times 312$ 空间信息在特定矩阵投影下坍缩成的一个全局解



## 结构支柱：`nn.Module` (神经网络的骨架)
在 PyTorch 中，所有的模型类都必须继承 `nn.Module`。

* **`super(Model, self).__init__()`**：
    * 这行代码是在向父类“申请初始化”。
    * 它会自动激活 PyTorch 所有的底层魔法（如参数管理、自动求导监控等）。
* **`self.bert` 与 `self.fc`**：
    * 这就是在定义“零件”。
    * 你把预训练好的 `BertModel` 当成一个现成的“高级搜索引擎”，把自定义的线性层 `nn.Linear` 当成“最后的翻译官”。


## 导致 temperature=0 失效的主要原因
### 1. 浮点数加法的“结合律失效”（最硬核的原因）
- 在简单的代数里，$ (A + B) + C = A + (B + C) $ 是绝对成立的真理。但在 GPU 极其暴力的并行计算中，为了快，几千个核心会同时把几万个数字加起来（也就是你刚才的 $\sum$ 操作）。谁先算完，谁就先加进总和里。问题在于，计算机存储浮点数（Float16 或 Bfloat16）的精度是有限的。如果计算顺序变了，在小数点后五六位就会产生极其微小的截断误差。在本地跑 BERT，可以通过 `deterministic=True` 强令 GPU：“必须按固定顺序算，宁可慢一点！”但商业大模型的服务器为了同时服务千万人，追求极致的吐字速度，绝对不可能开启这种牺牲性能的“强一致性锁”。所以，这种微小的浮点数误差是永远存在的。

### 2. MoE（混合专家）架构
- 目前的顶尖大模型几乎都采用了 MoE（Mixture of Experts）架构。当模型在推断下一个词时，会有一个“路由器”（Router）去给几十个“专家网络”打分，并挑选得分最高的两三个专家来干活。假设在极其复杂的矩阵运算后，专家 A 的得分是 $0.5000001$，专家 B 的得分是 $0.4999999$。此时，哪怕仅仅受到上面提到的第一点微小的浮点数误差干扰，下一次跑的时候，专家 B 的得分可能就变成了 $0.5000002$ 从而反超！一旦底层的“专家”换人了，整个网络生成的词就会瞬间改变，这就是为什么即使 $T=0$，大模型的回答也会突然变卦。

### 3. 动态批处理（Dynamic Batching）
- 调用 API 时，你其实不是一个人在独占那张显卡。为了省钱，服务器会把你的请求和世界上其他用户Batch到一起算。如 `pad_size=32`服务器为了把长短不一的句子拼成一个规整的超级大矩阵，会塞入大量的 [PAD]（零）。每次和你“Batch”的用户不同，矩阵的形状、填充的零的位置就会发生细微变化。这种内存对齐的差异，会再次诱发 GPU 底层算子产生微弱的数值波动。

## run.py具体细节

```python
parser = argparse.ArgumentParser(description='Chinese Text Classification')
parser.add_argument('--model', type=str, required=True, help='choose a model: bert, bert_tiny')
parser.add_argument('--data', type=str, required=True, help='choose a dataset: reject, intent')
args = parser.parse_args()
```

```python
if __name__ == '__main__':
    dataset = args.data
    model_name = args.model
    x = import_module('models.' + model_name)
    config = x.Config(dataset)
```
在终端输入`python run.py --model bert_tiny --data reject`

- dataset = args.data      # 此时 dataset 的值变成了字符串 "reject"
- model_name = args.model  # 此时 model_name 的值变成了字符串 "bert_tiny"
- x = import_module('models.' + model_name) #import models.bert_tiny as x
- config = models.bert_tiny.Config("reject")


In [1]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--name', type=str)
args = parser.parse_args(['--name', 'zhangxu'])
print(f"你好，{args.name}！这就是 parser 抓取到的值。")

你好，zhangxu！这就是 parser 抓取到的值。


  ```python
    np.random.seed(42) #封印 Numpy 的随机性。如果你的数据预处理、矩阵运算用到了 numpy.random，它保证每次生成的随机数序列一模一样。
    torch.manual_seed(42) #封印 PyTorch 在 CPU 上的随机性。保证 CPU 上初始化的权重、生成的随机张量每次都相同。
    torch.cuda.manual_seed_all(42) #封印 PyTorch 在 所有 GPU (显卡) 上的随机性。
    torch.backends.cudnn.deterministic = True #NVIDIA 的 cuDNN 底层加速库为了追求极致的计算速度，有时候会使用一些带随机性的算法（比如每次卷积计算的顺序可能不同，导致浮点数精度出现极其微小的误差）。True，相当于告诉显卡：“宁可牺牲一丁点计算速度，也要保证每次矩阵乘法算出来的结果精确到小数点后 8 位都绝对一致！”
```

# train_eval.py
## model 与 .bin 的关系
在程序运行过程中，`model` 和 `pytorch_model.bin` 经历了从“文件”到“生命”的过程。
### 1. pytorch_model.bin (硬盘上的静态文件)
* **本质**：它是保存在硬盘里的**二进制数值字典**。
* **状态**：它是“死”的数字。就像一本存放在图书馆里的百科全书，如果你不翻开它，它就只是一堆纸和油墨。
* **内容**：存储了 BERT 每一层神经元的权重（Weights）和偏置（Biases）。
### 2. model (内存中的活跃对象)
* **本质**：它是你在 Python 代码里通过 `BertModel` 类创建的**神经网络实例**。
* **状态**：它是“活”的结构。它已经占据了你的 GPU/CPU 内存，准备好接收数据并进行矩阵运算了。
* **关联**：当你执行 `model.train()` 或 `model(trains)` 时，操作的就是这个内存对象。

---

### 3.它们是如何互换的？（关键动作）

**动作 A：从文件到模型（加载知识）**
* **代码**：`BertModel.from_pretrained(config.bert_path)`
* **过程**：程序会去 `bert_path` 路径下找到那个 `.bin` 文件，把里面的几千万个数字读取出来，填充进 `model` 对象的每一个神经元里。

**动作 B：从模型到文件（保存成果）**
* **代码**：`torch.save(model.state_dict(), config.save_path)`
* **过程**：在 `train_eval.py` 中，当验证集表现最好时，程序会将 `model` 当前最聪明的“记忆”重新打包，写回硬盘变成一个新的 `.ckpt` 或 `.bin` 文件。

---

* **`pytorch_model.bin`** = 存放在 U 盘里的**游戏存档**。
* **`model`** = 正在电脑里**运行的游戏**。
* **加载** = 读档；**保存** = 存盘。

```python
    param_optimizer = list(model.named_parameters())
    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}]
```
### no_decay

`param_optimizer = list(model.named_parameters())`把模型内部所有需要学习的参数（权重和偏置）通通找出来，并给每一个参数配上它的“名字”

`no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']`列表里的参数在训练时不进行权重衰减
- bias（偏置项）：偏置项通常只是一个平移量，维数很小。对它进行衰减不仅没多大意义，反而可能导致模型拟合能力下降（欠拟合）。
- LayerNorm.bias 和 LayerNorm.weight：这是层归一化（Layer Normalization）的参数。它们的作用是保证神经元输出的分布稳定。如果强行让它们衰减（趋近于 0），会破坏模型各层之间的信号平衡。

### param
- `params`一个**“储物箱的标签”，告诉优化器：“这个箱子里装的全是你要负责更新的权重数据**。”告诉 BertAdam 优化器：“嘿，这一组参数（即列表里的所有 p）归你管，待会计算完梯度后，你要去修改它们的值。

```python
{
    'params': [这里是参数列表],
    'weight_decay': 0.01
}
```
`{'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01}`

- 筛选逻辑：如果参数的名字 $n$ 里面不包含 no_decay 里的那些关键词（比如没有 bias，没有 LayerNorm），就把它放进这一组。
- 分配政策：设置 weight_decay: 0.01。这意味着在训练时，系统会通过 $L_2$ 正则化稍微“压制”一下这些权重，不让它们长得太大，从而防止模型产生死记硬背过拟合。

### warmup
```python
    optimizer = BertAdam(optimizer_grouped_parameters,
                         lr=config.learning_rate,
                         warmup=0.05,
                         t_total=len(train_iter) * config.num_epochs)
```
- 'warmup' 在训练最开始的 $5\%$ 的步数里，学习率不会直接跳到 $5 \times 10^{-5}$，而是从 $0$ 开始缓慢爬升到最大值。模型刚开始练新数据时，梯度波动很大，慢启动能防止模型在第一轮就被带歪，让训练更稳。
- len(train_iter),在 PyTorch 中，train_iter 是一个数据加载器（DataLoader）。len(train_iter) 并不是指训练集里有多少句话，而是指有多少个“批次”（Batch）。模型每看一个 Batch，就会计算一次梯度，然后执行一次 optimizer.step() 来更新参数。len(train_iter) 就是跑完一遍全集（一个 Epoch）需要走的步数。
- BERT 训练通常采用线性衰减。即在过了预热期后，学习率要从最高点逐渐减小，直到训练结束时刚好降到接近 0。

### `model.train()`

model.train() 的核心作用是：告诉模型内部那些“性格分裂”的组件，现在要按“训练规则”还是“考试规则”来表现。它不负责“更新参数”
#### 1. 开启 Dropout（随机丢弃）
这是 BERT 类模型防止过拟合的“大招”。

- 在 model.train() 模式下：模型会随机让一部分神经元“断电”（置为 0）。这样模型就不能依赖某几个特定的神经元，被迫学到更健壮的特征。

- 在 model.eval() 模式下：所有的神经元全部恢复供电。如果训练时忘了切回 train，模型就会因为没有这种“负重训练”而产生严重的依赖，导致泛化能力变差。

#### 2. 控制 Batch Normalization（批归一化）
虽然你的 RoBERTa 模型主要用 LayerNorm，但理解这一点对深度学习非常重要：

- 在 model.train() 模式下：模型会根据**当前这一批数据（Batch）**的均值和方差来归一化，并且不断更新模型记录的“全局平均值”。

- 在 model.eval() 模式下：模型会停止更新，直接使用训练阶段攒下来的那个“全局平均值”来处理数据。

```python
outputs = model(trains)
model.zero_grad()
loss = F.cross_entropy(outputs, labels)
loss.backward()
optimizer.step()
```
### 训练全家桶
- outputs = model(trains)：预测。模型根据当前学到的知识，对输入数据给出一个答案。
- model.zero_grad()：归零。把上一次算错的账（梯度）清空，准备算这一次的。
- loss = F.cross_entropy(...)：判分。通过交叉熵算法，计算模型答案与标准答案之间的差距（Loss）。
- loss.backward()：归因。反向传播，算出每个神经元对这次“做错题”该负多少责任。
- optimizer.step()：改正。优化器根据责任大小，微调每个神经元的权重值。

```python
true = labels.data.cpu()
predic = torch.max(outputs.data, 1)[1].cpu()
train_acc = metrics.accuracy_score(true, predic)
dev_acc, dev_loss = evaluate(config, model, dev_iter)
if dev_loss < dev_best_loss:
```
- true = labels.data.cpu()：把标准答案（标签）从 GPU 拿回到 CPU。接下来的 metrics.accuracy_score 是 sklearn 库的函数，它不认识 GPU 上的数据，所以必须先执行 .cpu() 搬运回来。
- predic = torch.max(outputs.data, 1)[1].cpu()：outputs 是一个概率分布（比如：类别A占 10%，类别B占 80%）。torch.max(..., 1) 会找出每一行里概率最大的那个值。[0] 是最大概率的值。[1] 是这个最大值对应的索引（下标），也就是模型选出的类别编号。